# 02 - Ingeniería de Características (Pipeline B) y ZOI disjuntas
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. Pipeline B (camino principal)
El comité de tesis aprobó **Pipeline B** frente a una matriz frágil *una columna por curva numerada* (p ≫ n):

1. Features **por vuelta** en ZOI disjuntas (frenado | vértice | salida), con wrap-around en T1 / última curva.
2. Unidad de normalización: **tipología × sesión** (slow / medium / fast). Se resta la **mediana** del grid en esa celda.
3. Agregación **mediana** por piloto (media disponible como switch de sensibilidad). Se reporta `n_laps`.
4. Conjunto de 6–8 features (calidad > forzar 10): `brake_point_rel`, `trail_brake_frac`, `apex_speed_rel`, `throttle_attack`, `coast_frac`, `full_throttle_frac`, más `throttle_onset_rel` (ventana pre-apex) y `brake_point_iqr` post-agregación.

`config/circuits.yaml` deja de ser un archivo muerto: alimenta offsets ZOI, buckets de tipología y curvas clave.

Filtros canónicos FastF1: `pick_accurate ∩ pick_wo_box ∩ pick_not_deleted ∩ pick_track_status('1')`.  
`pick_quicklaps` es **push mode** explícito, no la definición por defecto de vuelta limpia.

*Las celdas de carga FastF1 requieren cache local; no se re-descarga telemetría en CI.*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_style_clustering.config import load_config, load_circuit_config
from f1_style_clustering.data.loader import load_session, enable_cache
from f1_style_clustering.features.zoi import extract_circuit_zones, disjoint_zone_bounds
from f1_style_clustering.features.aggregators import (
    build_pipeline_b_feature_matrix,
    CORE_STYLE_FEATURES,
)
from f1_style_clustering.features.preprocessor import standardize_features, separate_metadata
from f1_style_clustering.visualization.telemetry_plots import plot_corner_telemetry_comparison

cfg = load_config()
circuits = load_circuit_config()
print('Pipeline:', cfg['pipeline']['name'], '| aggregation:', cfg['features']['aggregation'])
print('ZOI offsets:', circuits['default_zoi_offsets'])
print('Core features:', CORE_STYLE_FEATURES)

# Opcional: descomentar con cache FastF1 local
# enable_cache('cache')
# session = load_session(year=2025, grand_prix='Italian Grand Prix', session_type='R')
# print('Sesión cargada:', session.event.EventName)


## 2. ZOI disjuntas y wrap-around
Las ventanas ya no se solapan (el frenado termina donde empieza el apex). T1 usa distancia de pista modular.


In [ ]:
# Demostración sin FastF1: T1 cerca de la línea de meta
bounds = disjoint_zone_bounds(
    apex_distance=40.0,
    braking_pre_m=350.0,
    apex_pre_m=60.0,
    apex_post_m=60.0,
    exit_post_m=250.0,
    track_length=5000.0,
)
print(bounds)

# Con sesión FastF1 en cache:
# analysis_zones = extract_circuit_zones(session, circuit_config=circuits)
# print(f'Total de curvas: {len(analysis_zones)}')


## 3. Matriz Pipeline B (6–8 features relativas)
`build_pipeline_b_feature_matrix` aplica filtros canónicos, features por (vuelta × curva × tipología), resta la mediana tipología×sesión y agrega por piloto.


In [ ]:
# Con sesión en cache:
# df_raw = build_pipeline_b_feature_matrix(session, circuit_config=circuits, aggregation='median')
# print(df_raw.shape)
# display(df_raw.head())
# df_scaled, df_meta, scaler = standardize_features(df_raw, scaler_type='robust')
print('Ver tests/test_pipeline_b.py para el contrato de features sin red.')
